In [13]:
############################################################

# EECE 5644 - Machine Learning
# Skylar Denno
# July 22, 2026

# ASSIGNMENT 6
# Long Short-Term Memory RNNs

############################################################

In [14]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

# pick the best avaliable device for training
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(device)

df = pd.read_csv('household_power_consumption.txt', sep=';', na_values='?')
print(df.head())

mps
         Date      Time  Global_active_power  Global_reactive_power  Voltage  \
0  16/12/2006  17:24:00                4.216                  0.418   234.84   
1  16/12/2006  17:25:00                5.360                  0.436   233.63   
2  16/12/2006  17:26:00                5.374                  0.498   233.29   
3  16/12/2006  17:27:00                5.388                  0.502   233.74   
4  16/12/2006  17:28:00                3.666                  0.528   235.68   

   Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  
0              18.4             0.0             1.0            17.0  
1              23.0             0.0             1.0            16.0  
2              23.0             0.0             2.0            17.0  
3              23.0             0.0             1.0            17.0  
4              15.8             0.0             1.0            17.0  


In [ ]:
######################
# DATA PREPROCESSING #
######################

#missing_idx = df.index[df.isna().any(axis=1)][0]
#print(df.iloc[missing_idx - 3:missing_idx + 4])

# Justification: For this problem, interpolation was chosen, as it will attempt to preserve the
# local trends seen in the data. This power data varies temporally, in short-term and long-term patterns.
# Additionally, the minute long sampling period makes the data change smoothly, and using forward or back fill 
# would add unnatural jumps in the data.

# add an absolute time index so interpolation follows the chronological order
df['DateTime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], dayfirst=True)
df = df.drop(columns=['Date', 'Time']).set_index('DateTime').sort_index()
numeric_cols = df.columns

missing_count = int(df[numeric_cols].isna().sum().sum())
print("Number of missing values found: ", missing_count)

# interpolate gaps from neighboring observations
df[numeric_cols] = df[numeric_cols].interpolate(method='time', limit_direction='both')

# fill ends of gaps (only) with forward and back fill
df[numeric_cols] = df[numeric_cols].ffill().bfill()

missing_after = int(df[numeric_cols].isna().sum().sum())
print("Number of missing values after filling: ", missing_after)

#print(df.iloc[missing_idx - 3:missing_idx + 4])




Number of missing values found. 181853
Number of missing values filled by interpolation. 0


In [ ]:
# Justification: Mean is used for volts, amps, power (kW) since they are continuously sampled,
# and the mean will represent the typical level over the hour.
# Summing is used for sub-metering channels because they represent
# energy usage contributions in watt-hours, so minute-to-minute totals add up.

agg_map = {
    'Global_active_power': 'mean',
    'Global_reactive_power': 'mean',
    'Voltage': 'mean',
    'Global_intensity': 'mean',
    'Sub_metering_1': 'sum',
    'Sub_metering_2': 'sum',
    'Sub_metering_3': 'sum'
}

df_h = df.resample('h').agg(agg_map)

# add cyclical calendar signals for daily and weekly seasonality
df_h['hour_sin'] = np.sin(2 * np.pi * df_h.index.hour / 24)
df_h['hour_cos'] = np.cos(2 * np.pi * df_h.index.hour / 24)
df_h['dow_sin'] = np.sin(2 * np.pi * df_h.index.dayofweek / 7)
df_h['dow_cos'] = np.cos(2 * np.pi * df_h.index.dayofweek / 7)

# add lag and rolling context features of the target for stronger autoregressive signal
df_h['gap_lag_24'] = df_h['Global_active_power'].shift(24)
df_h['gap_lag_168'] = df_h['Global_active_power'].shift(168)
df_h['gap_rollmean_24'] = df_h['Global_active_power'].rolling(24).mean()
df_h['gap_rollstd_24'] = df_h['Global_active_power'].rolling(24).std()
df_h['gap_rollmean_168'] = df_h['Global_active_power'].rolling(168).mean()

# Drop early rows that cannot support lag/rolling features
df_h = df_h.dropna().copy()

print(df_h.head())

# separate features and target
feature_cols = df_h.columns.tolist()
target_col = 'Global_active_power'
target_idx = feature_cols.index(target_col)
values = df_h[feature_cols].to_numpy(dtype=np.float32)

                     Global_active_power  Global_reactive_power     Voltage  \
DateTime                                                                      
2006-12-23 17:00:00             5.452533               0.215967  233.644167   
2006-12-23 18:00:00             3.879400               0.099767  238.000500   
2006-12-23 19:00:00             4.117833               0.205333  238.729333   
2006-12-23 20:00:00             4.181400               0.124767  238.518833   
2006-12-23 21:00:00             3.288433               0.235767  238.594667   

                     Global_intensity  Sub_metering_1  Sub_metering_2  \
DateTime                                                                
2006-12-23 17:00:00         23.360000           971.0            40.0   
2006-12-23 18:00:00         16.363333             0.0             1.0   
2006-12-23 19:00:00         17.300000             0.0            36.0   
2006-12-23 20:00:00         17.596667             0.0            21.0   
2006-12-

In [ ]:
##################
# MODEL TRAINING #
##################

history_hours = 7 * 24
forecast_hours = 24
stride_hours = 1

target_start_times = []
X_list, y_list = [], []
for start in range(0, len(values) - history_hours - forecast_hours + 1, stride_hours):
    x_window = values[start:start + history_hours, :]

    # Log-transform target to stabilize relative errors
    y_window_raw = values[start + history_hours:start + history_hours + forecast_hours, target_idx]
    y_window = np.log1p(np.clip(y_window_raw, a_min=0.0, a_max=None))

    X_list.append(x_window)
    y_list.append(y_window)
    target_start_times.append(df_h.index[start + history_hours])

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.float32)
target_start_times = pd.to_datetime(target_start_times)

print(f'Windowed samples: {len(X)} | X shape: {X.shape} | y shape: {y.shape}')

Windowed samples: 34230 | X shape: (34230, 168, 16) | y shape: (34230, 24)


In [43]:

# split off final 6 months for test set
cutoff_time = df_h.index.max() - pd.DateOffset(months=6)
test_mask = target_start_times >= cutoff_time
train_mask = target_start_times < cutoff_time

X_train_full, X_test = X[train_mask], X[test_mask]
y_train_full, y_test = y[train_mask], y[test_mask]

In [44]:
# create chronological validation split from the tail of the training data
val_frac = 0.15
val_size = max(1, int(len(X_train_full) * val_frac))

X_train, X_val = X_train_full[:-val_size], X_train_full[-val_size:]
y_train, y_val = y_train_full[:-val_size], y_train_full[-val_size:]

print(f'Train set: {len(X_train)}, validation set: {len(X_val)}, test set: {len(X_test)}')

Train set: 25361, validation set: 4475, test set: 4394


In [ ]:
# normalize using training statistics only
feat_mean = X_train.reshape(-1, X_train.shape[-1]).mean(axis=0)
feat_std = X_train.reshape(-1, X_train.shape[-1]).std(axis=0) + 1e-8

X_train_n = (X_train - feat_mean) / feat_std
X_val_n = (X_val - feat_mean) / feat_std
X_test_n = (X_test - feat_mean) / feat_std

# standardize y
y_mean = y_train.mean()
y_std = y_train.std() + 1e-8

y_train_n = (y_train - y_mean) / y_std
y_val_n = (y_val - y_mean) / y_std
y_test_n = (y_test - y_mean) / y_std

batch_size = 128
train_ds = TensorDataset(torch.tensor(X_train_n), torch.tensor(y_train_n))
val_ds = TensorDataset(torch.tensor(X_val_n), torch.tensor(y_val_n))
test_ds = TensorDataset(torch.tensor(X_test_n), torch.tensor(y_test_n))

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

In [46]:
class LSTM_inst(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, forecast_horizon=24, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.head = nn.Linear(hidden_size, forecast_horizon)

    def forward(self, x):
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]
        return self.head(last_hidden)

In [ ]:
def calc_val_mape(model, loader, y_mean, y_std, eps=0.1):
    model.eval()
    pred_list, true_list = [], []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            pred = model(xb).cpu().numpy()
            true = yb.cpu().numpy()
            pred_list.append(pred)
            true_list.append(true)

    pred_n = np.vstack(pred_list)
    true_n = np.vstack(true_list)

    # inverse standardization and inverse log1p transform
    pred_log = pred_n * y_std + y_mean
    true_log = true_n * y_std + y_mean
    pred_real = np.expm1(pred_log)
    true_real = np.expm1(true_log)

    pred_real = np.clip(pred_real, a_min=0.0, a_max=None)
    true_real = np.clip(true_real, a_min=0.0, a_max=None)

    return np.mean(np.abs((true_real - pred_real) / np.maximum(np.abs(true_real), eps))) * 100

In [48]:
#########################
# HYPERPARAMETER TUNING #
#########################

search_space = [
    {'hidden_size': 128, 'num_layers': 1, 'dropout': 0.0, 'lr': 5e-4},
    {'hidden_size': 128, 'num_layers': 2, 'dropout': 0.1, 'lr': 5e-4},
    {'hidden_size': 256, 'num_layers': 2, 'dropout': 0.2, 'lr': 3e-4},
    {'hidden_size': 256, 'num_layers': 3, 'dropout': 0.2, 'lr': 2e-4},
]

In [ ]:
class WeightedMAPELoss(nn.Module):
    def __init__(self, y_mean, y_std, eps=0.1):
        super().__init__()
        self.y_mean = float(y_mean)
        self.y_std = float(y_std)
        self.eps = eps

    def forward(self, pred_z, true_z):
        pred_log = pred_z * self.y_std + self.y_mean
        true_log = true_z * self.y_std + self.y_mean

        pred_real = torch.expm1(pred_log)
        true_real = torch.expm1(true_log)

        pred_real = torch.clamp(pred_real, min=0.0)
        true_real = torch.clamp(true_real, min=0.0)

        denom = torch.clamp(torch.abs(true_real), min=self.eps)
        ape = torch.abs(pred_real - true_real) / denom
        return ape.mean()

In [ ]:

epochs = 30
criterion = WeightedMAPELoss(y_mean=y_mean, y_std=y_std, eps=0.1)
patience = 6
min_delta = 0.02

best_config = None
best_val_mape = float('inf')
best_state_dict = None

for i, cfg in enumerate(search_space, start=1):
    model = LSTM_inst(
        input_size=len(feature_cols),
        hidden_size=cfg['hidden_size'],
        num_layers=cfg['num_layers'],
        forecast_horizon=forecast_hours,
        dropout=cfg['dropout']
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=1e-4)

    local_best_mape = float('inf')
    local_best_state = None
    wait = 0

    for epoch in range(1, epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        val_mape = calc_val_mape(model, val_loader, y_mean, y_std, eps=0.1)

        if val_mape < (local_best_mape - min_delta):
            local_best_mape = val_mape
            local_best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            break

    print(f"Config {i}: {cfg} | best val MAPE: {local_best_mape:.2f}%")

    if local_best_mape < best_val_mape:
        best_val_mape = local_best_mape
        best_config = cfg
        best_state_dict = local_best_state

print(f"Best config: {best_config} | best val MAPE: {best_val_mape:.2f}%")

Config 1: {'hidden_size': 128, 'num_layers': 1, 'dropout': 0.0, 'lr': 0.0005} | best val MAPE: 42.66%
Config 2: {'hidden_size': 128, 'num_layers': 2, 'dropout': 0.1, 'lr': 0.0005} | best val MAPE: 43.08%
Config 3: {'hidden_size': 256, 'num_layers': 2, 'dropout': 0.2, 'lr': 0.0003} | best val MAPE: 42.95%
Config 4: {'hidden_size': 256, 'num_layers': 3, 'dropout': 0.2, 'lr': 0.0002} | best val MAPE: 43.57%
Best config: {'hidden_size': 128, 'num_layers': 1, 'dropout': 0.0, 'lr': 0.0005} | best val MAPE: 42.66%


In [ ]:
#################
# MODEL TESTING #
#################

# load best model from tuning
best_model = LSTM_inst(
    input_size=len(feature_cols),
    hidden_size=best_config['hidden_size'],
    num_layers=best_config['num_layers'],
    forecast_horizon=forecast_hours,
    dropout=best_config['dropout']
).to(device)

best_model.load_state_dict(best_state_dict)
best_model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        pred = best_model(xb).cpu().numpy()
        true = yb.cpu().numpy()
        all_preds.append(pred)
        all_true.append(true)

In [ ]:
# evaluate test run in original target units

pred_n = np.vstack(all_preds)
true_n = np.vstack(all_true)

# Inverse standardization and inverse log1p.
pred_log = pred_n * y_std + y_mean
true_log = true_n * y_std + y_mean

pred = np.expm1(pred_log)
true = np.expm1(true_log)

pred = np.clip(pred, a_min=0.0, a_max=None)
true = np.clip(true, a_min=0.0, a_max=None)

mae = np.mean(np.abs(pred - true))
rmse = np.sqrt(np.mean((pred - true) ** 2))
mape = np.mean(np.abs((true - pred) / np.maximum(np.abs(true), 0.1))) * 100

print(f'Test MAE ({target_col}): {mae:.4f}')
print(f'Test RMSE ({target_col}): {rmse:.4f}')
print(f'Test MAPE_eps0.1 ({target_col}): {mape:.2f}%')

Test MAE (Global_active_power): 0.4485
Test RMSE (Global_active_power): 0.6818
Test MAPE_eps0.1 (Global_active_power): 41.28%


In [ ]:
# next day forecast using the best model
last_window = values[-history_hours:, :]
last_window_n = (last_window - feat_mean) / feat_std
last_window_t = torch.tensor(last_window_n).unsqueeze(0).to(device)

with torch.no_grad():
    next24_n = best_model(last_window_t).cpu().numpy().reshape(-1)

# Inverse standardization and inverse log1p to original units
next24_log = next24_n * y_std + y_mean
next24 = np.expm1(next24_log)
next24 = np.clip(next24, a_min=0.0, a_max=None)

forecast_index = pd.date_range(
    start=df_h.index[-1] + pd.Timedelta(hours=1),
    periods=forecast_hours,
    freq='h'
)

next24_df = pd.DataFrame({'predicted_global_active_power': next24}, index=forecast_index)

print('\nNext 24-hour forecast:')
print(next24_df.head(24))


Next 24-hour forecast:
                     predicted_global_active_power
2010-11-26 22:00:00                       0.595620
2010-11-26 23:00:00                       0.431990
2010-11-27 00:00:00                       0.381578
2010-11-27 01:00:00                       0.333676
2010-11-27 02:00:00                       0.355474
2010-11-27 03:00:00                       0.341529
2010-11-27 04:00:00                       0.292817
2010-11-27 05:00:00                       0.428865
2010-11-27 06:00:00                       0.660410
2010-11-27 07:00:00                       1.155801
2010-11-27 08:00:00                       1.384477
2010-11-27 09:00:00                       1.563559
2010-11-27 10:00:00                       1.252036
2010-11-27 11:00:00                       1.238327
2010-11-27 12:00:00                       1.223849
2010-11-27 13:00:00                       1.185802
2010-11-27 14:00:00                       1.111205
2010-11-27 15:00:00                       1.031468
2010-11

In [54]:
####################
# ANOMOLY ALERTING #
####################


def predict_original_units(model, X_norm, y_mean, y_std, batch_size=512):
    ds = TensorDataset(torch.tensor(X_norm))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    preds = []
    model.eval()
    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(device)
            pred_z = model(xb).cpu().numpy()
            pred_log = pred_z * y_std + y_mean
            pred_real = np.expm1(pred_log)
            pred_real = np.clip(pred_real, a_min=0.0, a_max=None)
            preds.append(pred_real)
    return np.vstack(preds)

In [55]:
def two_hour_alerts(residual_series, threshold):
    # true at hours where |residual| crosses threshold
    exceed = np.abs(residual_series) > threshold

    # mark both hours in any 2-consecutive exceedance pair
    alerts = np.zeros_like(exceed, dtype=bool)
    if len(exceed) > 1:
        pair = exceed[1:] & exceed[:-1]
        alerts[1:] |= pair
        alerts[:-1] |= pair
    return alerts



In [56]:

# horizon-1 residuals so each timestamp is unique
train_pred = predict_original_units(best_model, X_train_n, y_mean, y_std)
val_pred = predict_original_units(best_model, X_val_n, y_mean, y_std)
test_pred = predict_original_units(best_model, X_test_n, y_mean, y_std)

train_true = np.expm1(y_train_n * y_std + y_mean)
val_true = np.expm1(y_val_n * y_std + y_mean)
test_true = np.expm1(y_test_n * y_std + y_mean)

train_resid_h1 = (train_true[:, 0] - train_pred[:, 0])
val_resid_h1 = (val_true[:, 0] - val_pred[:, 0])
test_resid_h1 = (test_true[:, 0] - test_pred[:, 0])

resid_std = float(np.std(train_resid_h1))
base_k = 3.0



In [57]:
# calibrate k on validation so false-alarm rate stays below 5%
best_k = base_k
val_far_pct = None
for k in np.arange(base_k, 8.01, 0.25):
    threshold_k = k * resid_std
    val_alerts_k = two_hour_alerts(val_resid_h1, threshold_k)
    far_k = val_alerts_k.mean() * 100
    if far_k <= 5.0:
        best_k = float(k)
        val_far_pct = float(far_k)
        break

# if no k in range meets 5%, keep the strictest tested threshold
if val_far_pct is None:
    best_k = 8.0
    threshold_k = best_k * resid_std
    val_alerts_k = two_hour_alerts(val_resid_h1, threshold_k)
    val_far_pct = float(val_alerts_k.mean() * 100)

threshold = best_k * resid_std

In [58]:

# TEST the system
test_alerts = two_hour_alerts(test_resid_h1, threshold)
test_alert_rate_pct = float(test_alerts.mean() * 100)

# Timestamps corresponding to horizon-1 targets for each sample
test_times = pd.DatetimeIndex(target_start_times[test_mask])
alert_times = test_times[test_alerts]

print(f"Residual std (train, horizon-1): {resid_std:.4f}")
print(f"Threshold multiplier k: {best_k:.2f}")
print(f"Alert threshold |residual| > {threshold:.4f}")
print(f"Validation false-alarm rate: {val_far_pct:.2f}% (target < 5%)")
print(f"Test alert rate: {test_alert_rate_pct:.2f}%")
print(f"Number of flagged hours in test: {int(test_alerts.sum())}")

if len(alert_times) > 0:
    print("First 10 alert timestamps:")
    print(alert_times[:10])
else:
    print("No test alerts triggered under current threshold and rule.")

Residual std (train, horizon-1): 0.6117
Threshold multiplier k: 3.00
Alert threshold |residual| > 1.8350
Validation false-alarm rate: 0.34% (target < 5%)
Test alert rate: 0.09%
Number of flagged hours in test: 4
First 10 alert timestamps:
DatetimeIndex(['2010-06-05 22:00:00', '2010-06-05 23:00:00',
               '2010-07-14 22:00:00', '2010-07-14 23:00:00'],
              dtype='datetime64[ns]', freq=None)
